# EDSS 취업데이터 2022–2023 학교·학과 수 기반 연결

## tl;dr

학생 수를 쓰지 않는다. 같은 지역과 대학/대학원 구분 안에서 2022 개방ID별 고유 학과 수와 2023 학교명별 고유 학과 수의 차이가 0~2인 후보를 만들고, 학과명 교집합으로 후보를 정렬한다. 후보는 검토용이며 확정 매핑으로 쓰지 않는다.

## Context & Methods

- 2022 학교 단위: `개방ID`
- 2023 학교 단위: `(시도명, 학교명, 대학대학원구분명)`
- 지역·대학/대학원 구분이 같은 후보만 비교
- 규모 지표: 공백과 전각/반각을 정규화한 `학과명`의 고유 개수
- 후보 조건: 학과 수 절대차가 2 이하
- 순위: 학과명 교집합 수, Jaccard, 작은 집합 포괄률, 학과 수 차이 순
- 결과표에는 후보를 학교당 최대 5개만 기록하며 원본은 수정하지 않음

In [1]:
from pathlib import Path
import csv
import hashlib
import json
import subprocess
import pandas as pd

ROOT = Path('/Users/joocheol/Documents/ChatGPT/EDSS')
RAW_ROOT = Path('/Users/joocheol/Documents/GitHub/edss/data/raw/edss')
SCRIPT = ROOT / 'scripts/match_edss_employment_2022_2023_school_counts.py'
SUMMARY = ROOT / 'data/metadata/edss_employment_2022_2023_department_count_match.json'
REGION_COUNTS = ROOT / 'data/metadata/edss_employment_2022_2023_region_school_counts.csv'
CANDIDATES = ROOT / 'data/processed/edss/restricted/derived/employment_2022_2023_department_count_candidates.csv'

## Data

원본 ZIP 두 개와 기존 학교-연도 브리지의 체크섬을 기록한 뒤 분석을 다시 실행한다.

In [2]:
run = subprocess.run(
    ['python3', str(SCRIPT), '--raw-root', str(RAW_ROOT)],
    cwd=ROOT, check=True, capture_output=True, text=True,
)
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
region = pd.read_csv(REGION_COUNTS, encoding='utf-8-sig')
summary['inputs']

{'source_2022': {'path': '/Users/joocheol/Documents/GitHub/edss/data/raw/edss/취업통계/0001_학생인적취업정보_13299/0001_학생인적취업정보_2022.zip',
  'sha256': 'e763b6a6848039be6c1e34975628600f772e36219a762610f8a6a59a1b694d13'},
 'source_2023': {'path': '/Users/joocheol/Documents/GitHub/edss/data/raw/edss/취업통계/0001_학생인적취업정보_13300/0001_학생인적취업정보_2023.zip',
  'sha256': '1ace9e82663a816e00a66a35d2b76980e79c708d54abd390564d3d90dcdbe4eb'},
 'bridge': {'path': 'data/metadata/edss_school_year_bridge.csv',
  'sha256': 'edfbdbf2cc9c20da00f47d059d668eb8fa4469fae79c6c3105bb356ab644a27d'}}

## Results

지역별 학교 수 비교와 후보 판정 상태를 집계만 표시한다.

In [3]:
pd.DataFrame(summary['counts']['region_level_school_counts']).T.rename(columns={
    'region_count': '지역 수',
    'school_appearances_2022': '2022 학교 출현 수',
    'school_identities_2023': '2023 학교 단위 수',
    'regions_with_absolute_difference_at_most_tolerance': '차이 2 이하 지역 수',
    'maximum_absolute_difference': '지역별 최대 절대차',
})

,지역 수,2022 학교 출현 수,2023 학교 단위 수,차이 2 이하 지역 수,지역별 최대 절대차
대학,17,386,415,14,16
대학원,17,183,1204,0,382


In [4]:
region.assign(절대차=region['difference_2023_minus_2022'].abs()).sort_values(
    ['level', '절대차'], ascending=[True, False]
).rename(columns={
    'province': '지역', 'level': '구분', 'schools_2022': '2022 학교 수',
    'schools_2023': '2023 학교 수', 'difference_2023_minus_2022': '2023-2022'
})

,지역,구분,2022 학교 수,2023 학교 수,2023-2022,절대차
16,서울,대학,54,70,16,16
6,경북,대학,36,40,4,4
2,경기,대학,70,73,3,3
12,대전,대학,17,19,2,2
10,대구,대학,13,14,1,1
14,부산,대학,24,25,1,1
26,전북,대학,20,21,1,1
30,충남,대학,26,27,1,1
0,강원,대학,21,21,0,0
4,경남,대학,23,23,0,0


In [5]:
pd.DataFrame(summary['counts']['level_status_counts']).fillna(0).astype(int).T

,ambiguous_top_score_tie,no_count_tolerance_candidate,unique_best_review_required,unique_exact_department_set,unique_high_overlap,reverse_open_id_conflict
대학,14,55,278,44,24,0
대학원,357,208,614,13,8,4


### Validation

후보 파일의 행 수·학교 단위 유일성·체크섬과 확정 매핑 미작성 조건을 검증한다.

In [6]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

with CANDIDATES.open(encoding='utf-8-sig', newline='') as handle:
    rows = list(csv.DictReader(handle))
top_rows = [row for row in rows if row['candidate_rank'] in ('', '1')]
assert len(rows) == summary['counts']['candidate_output_row_count']
assert len(top_rows) == summary['counts']['school_identity_count_2023']
assert len({row['school_identity_key'] for row in top_rows}) == len(top_rows)
assert sha256(CANDIDATES) == summary['outputs']['candidate_output']['sha256']
assert sha256(REGION_COUNTS) == summary['outputs']['region_count_output']['sha256']
assert summary['method']['canonical_mapping_written'] is False
{'candidate_rows': len(rows), 'school_identities': len(top_rows), 'validation': 'PASS'}

{'candidate_rows': 4625, 'school_identities': 1619, 'validation': 'PASS'}

## Takeaways

- 대학은 17개 지역 중 14개에서 지역별 학교 수 차이가 2 이하라서 제안한 가정이 대체로 성립한다.
- 대학원은 2023 학교명이 개별 대학원·과정 수준으로 더 세분되어 17개 지역 모두 학교 수 차이가 2를 넘는다.
- 따라서 대학은 이 후보표로 직접 검토할 수 있지만, 대학원은 먼저 동일 대학 본체 아래의 대학원명을 묶는 상위학교 정규화가 필요하다.
- 정확 일치와 높은 학과명 중첩도 자동 확정이 아니라 검토 우선순위일 뿐이다.